In [ ]:

!pip install opencv-python


EXTRACTING DATA
::::::::::::::::::::::::::::::
::::::::::::::::::::::::::::::

In [ ]:
#extract from dataset
import os

your_dataset = []
base_folder = "GTSRB/Train"  # path to your Train folder

for label_folder in os.listdir(base_folder):
    label_path = os.path.join(base_folder, label_folder)
    if os.path.isdir(label_path):
        for filename in os.listdir(label_path):
            if filename.endswith(".ppm") or filename.endswith(".png") or filename.endswith(".jpg"):
                img_path = os.path.join(label_path, filename)
                your_dataset.append((img_path, int(label_folder)))  # label is the folder name as integer




HOG LOGISTIC REGRESSION ON NORMAL DATA
:::::::::::::::::::::::::::::::::
:::::::::::::::::::::::::::::::::

In [ ]:
from skimage.feature import hog
from skimage import color
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import cv2
import os

# Load and preprocess dataset
X = []
y = []

# Replace with your actual dataset loading logic
for img_path, label in your_dataset:
    img = cv2.imread(img_path)
    img = cv2.resize(img, (64, 64))
    gray = color.rgb2gray(img)

    # Extract HOG features
    features = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys')
    
    X.append(features)
    y.append(label)

X = np.array(X)
y = np.array(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"HOG + Logistic Regression Accuracy: {acc:.4f}")


HOG LR TEST
:::::::::::::::;
:::::::::::::::::

In [ ]:
# needed
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2
from skimage.feature import hog  # assuming skimage is installed
from skimage import exposure
# Label map (minimal or full version; update as needed)
label_map = {
    0: "Speed limit 20", 1: "Speed limit 30", 2: "Speed limit 50", 3: "Speed limit 60", 4: "Speed limit 70",
    5: "Speed limit 80", 6: "End limit 80", 7: "Speed limit 100", 8: "Speed limit 120", 9: "No overtaking",
    10: "No overtaking (trucks)", 11: "Right of way", 12: "Yield", 13: "Stop", 14: "No vehicles",
    15: "No trucks", 16: "No entry", 17: "Danger", 18: "Left curve", 19: "Right curve",
    20: "Double curve", 21: "Uneven road", 22: "Slippery", 23: "Road narrows", 24: "Construction",
    25: "Signal", 26: "Pedestrian", 27: "Children", 28: "Bicycles", 29: "Ice/Snow",
    30: "Animals", 31: "End restrictions", 32: "Turn right", 33: "Turn left", 34: "Go straight",
    35: "Go straight/right", 36: "Go straight/left", 37: "Keep right", 38: "Keep left", 39: "Roundabout",
    40: "End no overtaking", 41: "End no overtaking (trucks)", 42: "Other"
}
class_names = [label_map[i] for i in range(43)]



# --- Predict on External Images using HOG ---
test_folder = "../RoadsignRecognition/GTSRB/Test"
print("Individual Image Predictions:\n")
for filename in os.listdir(test_folder):
    if filename.lower().endswith((".ppm", ".png", ".jpg")):
        img_path = os.path.join(test_folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (64, 64))

        # Extract HOG features (same as used during training)
        hog_features = hog(img, orientations=9, pixels_per_cell=(8, 8),
                           cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)

        predicted_label = model.predict([hog_features])[0]
        print(f"Image: {filename}, Predicted Label: {predicted_label}, Class: {label_map[predicted_label]}")

print()

# --- Confusion Matrix ---
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# --- Per-Class Accuracy ---
correct_per_class = cm.diagonal()
total_per_class = cm.sum(axis=1)
class_accuracies = correct_per_class / total_per_class

plt.figure(figsize=(14, 5))
sns.barplot(x=class_names, y=class_accuracies)
plt.title("Per-Class Accuracy")
plt.ylabel("Accuracy")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


SVM ON NORMAL DATA
:::::::::::::::::::
:::::::::::::::::::

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
import numpy as np
import cv2
import os
import seaborn as sns
import matplotlib.pyplot as plt

X = []  
y = []  

# Replace this with your actual dataset loading
for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    X.append(img.flatten() / 255.0)
    y.append(label)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = SVC(kernel='rbf', C=10, gamma='scale')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Macro and Weighted Scores
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')

precision_weighted = precision_score(y_test, y_pred, average='weighted')
recall_weighted = recall_score(y_test, y_pred, average='weighted')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

print(f"\nWeighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")


SVM NORMAL TEST
:::::::::::::::
:::::::::::::::

In [ ]:
# needed
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2

# Label map (minimal or full version; update as needed)
label_map = {
    0: "Speed limit 20", 1: "Speed limit 30", 2: "Speed limit 50", 3: "Speed limit 60", 4: "Speed limit 70",
    5: "Speed limit 80", 6: "End limit 80", 7: "Speed limit 100", 8: "Speed limit 120", 9: "No overtaking",
    10: "No overtaking (trucks)", 11: "Right of way", 12: "Yield", 13: "Stop", 14: "No vehicles",
    15: "No trucks", 16: "No entry", 17: "Danger", 18: "Left curve", 19: "Right curve",
    20: "Double curve", 21: "Uneven road", 22: "Slippery", 23: "Road narrows", 24: "Construction",
    25: "Signal", 26: "Pedestrian", 27: "Children", 28: "Bicycles", 29: "Ice/Snow",
    30: "Animals", 31: "End restrictions", 32: "Turn right", 33: "Turn left", 34: "Go straight",
    35: "Go straight/right", 36: "Go straight/left", 37: "Keep right", 38: "Keep left", 39: "Roundabout",
    40: "End no overtaking", 41: "End no overtaking (trucks)", 42: "Other"
}
class_names = [label_map[i] for i in range(43)]

# --- Predict on External Images ---
test_folder = "../RoadsignRecognition/GTSRB/Test"
print("Individual Image Predictions:\n")
for filename in os.listdir(test_folder):
    if filename.lower().endswith((".ppm", ".png", ".jpg")):
        img_path = os.path.join(test_folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (64,64))
        img_flattened = img.flatten() / 255.0
        predicted_label = model.predict([img_flattened])[0]
        print(f"Image: {filename}, Predicted Label: {predicted_label}, Class: {label_map[predicted_label]}")
print()

# --- Confusion Matrix ---
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# --- Per-Class Accuracy ---
correct_per_class = cm.diagonal()
total_per_class = cm.sum(axis=1)
class_accuracies = correct_per_class / total_per_class

plt.figure(figsize=(14, 5))
sns.barplot(x=class_names, y=class_accuracies)
plt.title("Per-Class Accuracy")
plt.ylabel("Accuracy")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


SVM MODEL OLD
::::::::::::::
::::::::::::::::

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import cv2
import os
import seaborn as sns
import matplotlib.pyplot as plt


X = []  
y = []  

for img_path, label in your_dataset:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    X.append(img.flatten() / 255.0)
    y.append(label)

X = np.array(X)
y = np.array(y)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = SVC(kernel='poly', degree=3,C=10, gamma='scale')  
model.fit(X_train, y_train)


y_pred = model.predict(X_test)


print("Accuracy:", accuracy_score(y_test, y_pred))


print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(43)]))


cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(43), yticklabels=np.arange(43))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


!!!hog svm

In [ ]:
import os
import cv2
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, precision_score,
    recall_score, f1_score, confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt

# ---------------------------
# CONFIG
# ---------------------------
DATA_DIR = 'GTSRB/Train'  # your dataset folder

# ---------------------------
# INIT
# ---------------------------
hog = cv2.HOGDescriptor()
X, y = [], []

# ---------------------------
# LOAD DATA
# ---------------------------
class_folders = [f for f in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, f))]
class_folders = sorted(class_folders)
print("Classes used:", class_folders)

for class_idx, class_name in enumerate(class_folders):
    class_dir = os.path.join(DATA_DIR, class_name)
    for img_name in os.listdir(class_dir):
        img_path = os.path.join(class_dir, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, (64, 128))
        features = hog.compute(img).flatten()
        X.append(features)
        y.append(class_idx)

X = np.array(X)
y = np.array(y)
print(f"Total samples: {len(X)}")

# ---------------------------
# SPLIT DATA
# ---------------------------
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

# ---------------------------
# TRAIN SVM
# ---------------------------
clf = svm.SVC(kernel='rbf')
clf.fit(X_train, y_train)

# ---------------------------
# PREDICT
# ---------------------------
y_train_pred = clf.predict(X_train)
y_val_pred = clf.predict(X_val)

# ---------------------------
# TRAINING ACCURACY
# ---------------------------
train_acc = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {train_acc:.4f}")

# ---------------------------
# VALIDATION ACCURACY
# ---------------------------
val_acc = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {val_acc:.4f}")

# ---------------------------
# DETAILED METRICS
# ---------------------------
precision_macro = precision_score(y_val, y_val_pred, average='macro')
recall_macro = recall_score(y_val, y_val_pred, average='macro')
f1_macro = f1_score(y_val, y_val_pred, average='macro')

precision_weighted = precision_score(y_val, y_val_pred, average='weighted')
recall_weighted = recall_score(y_val, y_val_pred, average='weighted')
f1_weighted = f1_score(y_val, y_val_pred, average='weighted')

print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

print(f"\nWeighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")

# ---------------------------
# CLASSIFICATION REPORT
# ---------------------------
print("\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=class_folders))

# ---------------------------
# CONFUSION MATRIX
# ---------------------------
cm = confusion_matrix(y_val, y_val_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_folders,
            yticklabels=class_folders)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()
